# Video → scene.glb + flythrough.mp4 (VGGT)

This notebook turns **your uploaded video** into:
1. `scene.glb` — interactive 3D of that video’s space
2. `flythrough.mp4` — moving camera through that 3D

## Setup
1. Runtime → **T4 GPU**
2. Run Cell A
3. **Runtime → Restart session**
4. Run Cell B onward


In [ ]:
# Cell A — install (then RESTART SESSION)
import os, sys, shutil
SRC='/content/vggt_src'
if os.path.isdir('/content/vggt') and not os.path.isfile('/content/vggt/models/vggt.py'):
    shutil.rmtree('/content/vggt')
if not os.path.isdir(f'{SRC}/vggt/models'):
    !git clone --depth 1 https://github.com/facebookresearch/vggt.git {SRC}
!{sys.executable} -m pip -q uninstall -y numpy
!{sys.executable} -m pip -q install "numpy==1.26.4"
!{sys.executable} -m pip -q install Pillow huggingface_hub einops safetensors opencv-python-headless trimesh matplotlib scipy tqdm imageio imageio-ffmpeg
!{sys.executable} -m pip -q install -e {SRC}
print('Restart session now, then run Cell B')


In [ ]:
# Cell B — imports after restart
import sys
SRC='/content/vggt_src'
sys.path.insert(0, SRC)
import numpy as np, torch
print('numpy', np.__version__)
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
print('OK')


In [ ]:
# Cell C — upload THE SAME video from Streamlit
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print('Video:', VIDEO_PATH)


In [ ]:
# Cell D — frames from that video
import cv2
from pathlib import Path
frames_dir=Path('/content/frames'); frames_dir.mkdir(exist_ok=True)
for p in frames_dir.glob('*'): p.unlink()
cap=cv2.VideoCapture(VIDEO_PATH)
fps=cap.get(cv2.CAP_PROP_FPS) or 30
interval=max(1,int(round(fps)))
max_frames=40; idx=saved=0
while saved<max_frames:
    ok,frame=cap.read()
    if not ok: break
    if idx%interval==0:
        cv2.imwrite(str(frames_dir/f'{saved:06d}.jpg'), frame); saved+=1
    idx+=1
cap.release()
image_names=sorted(str(p) for p in frames_dir.glob('*.jpg'))
print('frames', len(image_names))
assert len(image_names)>=2


In [ ]:
# Cell E — reconstruct THIS video
import sys, torch, numpy as np
SRC='/content/vggt_src'; sys.path.insert(0, SRC)
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

def to_numpy(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    if x.ndim >= 1 and x.shape[0] == 1:
        x = x.reshape(x.shape[1:])
    return x

device='cuda' if torch.cuda.is_available() else 'cpu'
assert device=='cuda', 'Enable GPU'
model=VGGT.from_pretrained('facebook/VGGT-1B').to(device).eval()
images=load_and_preprocess_images(image_names).to(device)
print('images', tuple(images.shape))
dtype=torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        predictions=model(images)

pose=predictions['pose_enc']
if pose.ndim==2:
    pose=pose.unsqueeze(0)
predictions['pose_enc']=pose
extrinsic, intrinsic = pose_encoding_to_extri_intri(pose, images.shape[-2:])
predictions['extrinsic']=extrinsic
predictions['intrinsic']=intrinsic
predictions['images']=images

for k,v in list(predictions.items()):
    if isinstance(v, torch.Tensor):
        predictions[k]=to_numpy(v)
predictions['pose_enc_list']=None
predictions['world_points_from_depth']=unproject_depth_map_to_point_map(
    predictions['depth'], predictions['extrinsic'], predictions['intrinsic'])
print('extrinsic', predictions['extrinsic'].shape)


In [ ]:
# Cell F — write scene.glb (interactive 3D)
import sys
SRC='/content/vggt_src'; sys.path.insert(0, SRC)
from visual_util import predictions_to_glb
scene=predictions_to_glb(predictions, conf_thres=50.0, filter_by_frames='all', show_cam=True, prediction_mode='Predicted Pointmap')
scene.export('/content/scene.glb')
print('Wrote /content/scene.glb')


In [ ]:
# Cell G — write flythrough.mp4 (moving path through the SAME reconstruction)
import numpy as np, cv2

if "world_points" in predictions:
    pts = np.asarray(predictions["world_points"]).reshape(-1, 3)
    conf = np.asarray(predictions.get("world_points_conf", np.ones(len(pts)))).reshape(-1)
else:
    pts = np.asarray(predictions["world_points_from_depth"]).reshape(-1, 3)
    conf = np.asarray(predictions.get("depth_conf", np.ones(len(pts)))).reshape(-1)

keep = conf >= np.percentile(conf, 60)
pts = pts[keep]
if len(pts) > 120000:
    pts = pts[np.random.default_rng(0).choice(len(pts), 120000, replace=False)]

extrinsics = np.asarray(predictions["extrinsic"])  # (S,3,4)
center = np.median(pts, axis=0)
scale = np.percentile(np.linalg.norm(pts - center, axis=1), 90) + 1e-6
pts_n = (pts - center) / scale

W, H = 720, 480
out = cv2.VideoWriter("/content/flythrough.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 12.0, (W, H))

def render(E):
    R, t = E[:, :3], E[:, 3]
    tn = (R @ center + t) / scale
    Xc = (R @ pts_n.T).T + tn
    z = Xc[:, 2]
    valid = z > 0.05
    frame = np.full((H, W, 3), 18, np.uint8)
    if valid.sum() < 50:
        return frame
    u, v = Xc[valid, 0] / z[valid], Xc[valid, 1] / z[valid]
    u0, u1 = np.percentile(u, [5, 95]); v0, v1 = np.percentile(v, [5, 95])
    u1 = u0 + 1e-3 if u1 - u0 < 1e-3 else u1
    v1 = v0 + 1e-3 if v1 - v0 < 1e-3 else v1
    px = np.clip(((u - u0) / (u1 - u0) * (W - 1)).astype(np.int32), 0, W - 1)
    py = np.clip(((v - v0) / (v1 - v0) * (H - 1)).astype(np.int32), 0, H - 1)
    depth = z[valid]
    col = (255 * (1.0 - (depth - depth.min()) / (np.ptp(depth) + 1e-6))).astype(np.uint8)
    frame[py, px, 0] = col
    frame[py, px, 1] = col
    frame[py, px, 2] = np.clip(col.astype(np.int32) + 40, 0, 255).astype(np.uint8)
    return frame

for i in range(len(extrinsics) - 1):
    for a in np.linspace(0, 1, 4, endpoint=False):
        out.write(render((1 - a) * extrinsics[i] + a * extrinsics[i + 1]))
out.write(render(extrinsics[-1]))
out.release()
print("Wrote /content/flythrough.mp4")



In [ ]:
# Cell H — download BOTH files → upload them in Streamlit for this job
from google.colab import files
files.download('/content/scene.glb')
files.download('/content/flythrough.mp4')
print('In Streamlit: upload scene.glb and flythrough.mp4 for your job')
